# 01 - Validation on the Curated Control Set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-pipeline/blob/main/notebooks/01_validation_colab.ipynb)

Walks through `crypticip download-validation` and
`crypticip validate --config config/validation.yaml` on the curated 9-
structure control set (~10 MB download). Without external binaries the
validate step exits in fallback mode — useful for sanity-checking the
pipeline, but **not** for scientific conclusions.


## Run this first — fresh-Colab setup

Configure the variables below, then run every cell in this section in
order. Re-running is safe: the clone is updated in place and pip
installs are idempotent.

- `REPO_URL` / `BRANCH` — where to fetch the code from.
- `PROJECT_DIR` — where to clone into (`/content/...` lives on the
  Colab VM and is wiped between sessions).
- `MOUNT_DRIVE` — set `True` to mount Google Drive so outputs persist.
- `DRIVE_RESULTS` — if mounting, where on Drive to put `results/`.


In [ ]:
REPO_URL    = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
BRANCH      = 'main'
PROJECT_DIR = '/content/cryptic-ip-pipeline'
MOUNT_DRIVE = False                  # True to mount Google Drive
DRIVE_ROOT  = '/content/drive'
DRIVE_RESULTS = '/content/drive/MyDrive/crypticip/results'  # used only if MOUNT_DRIVE


In [ ]:
import os, subprocess, sys, pathlib

project = pathlib.Path(PROJECT_DIR)
project.parent.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

if (project / '.git').exists():
    sh(['git', '-C', str(project), 'fetch', '--quiet', 'origin', BRANCH])
    sh(['git', '-C', str(project), 'checkout', BRANCH])
    sh(['git', '-C', str(project), 'reset', '--hard', f'origin/{BRANCH}'])
else:
    sh(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, str(project)])

os.chdir(project)
print('cwd:', os.getcwd())
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat'])


In [ ]:
# Optional: mount Drive and redirect results/ onto it so outputs survive
# a runtime swap. Safe to re-run; pre-existing local results/ is backed
# up to a timestamped sibling rather than deleted.
import datetime, pathlib, shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)
    drive_path = pathlib.Path(DRIVE_RESULTS)
    drive_path.mkdir(parents=True, exist_ok=True)
    target = pathlib.Path(PROJECT_DIR) / 'results'
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        backup = target.with_name(f'results.local_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}')
        shutil.move(str(target), str(backup))
        print('existing results/ backed up to:', backup)
    target.symlink_to(drive_path, target_is_directory=True)
    print('results/ ->', drive_path)
else:
    print('MOUNT_DRIVE=False; outputs will live on the Colab VM only.')


In [ ]:
# Verify the CLI is wired up correctly.
!crypticip --version
!crypticip check-env
!crypticip list-configs


## (Optional) Install external scientific binaries

Vanilla Colab does **not** ship `fpocket`, `freesasa`, `pdb2pqr`, `apbs`,
or `pymol`. The CLI runs in clearly-labelled fallback mode without them
(see `crypticip check-env`). For a **real** scientific run, install via
condacolab + mamba. This **restarts the kernel** — after the restart,
you must re-run the bootstrap cell above before continuing.

Skip this section entirely if you only want to exercise the plumbing.


In [ ]:
INSTALL_EXTERNAL = False  # set True to install fpocket/apbs/pymol via condacolab

if INSTALL_EXTERNAL:
    !pip install -q condacolab
    import condacolab
    condacolab.install()   # >>> kernel restarts here <<<


In [ ]:
# Post-restart cell (only if you set INSTALL_EXTERNAL=True above).
# After the kernel restart, RE-RUN the bootstrap cells above (clone,
# pip install) before running this — Colab loses cwd and the editable
# install on restart.
INSTALL_EXTERNAL = False
if INSTALL_EXTERNAL:
    !mamba install -y -c conda-forge -c bioconda \
        fpocket freesasa pdb2pqr apbs pymol-open-source python-freesasa
    !crypticip check-env


## Download the validation set

Pulls 5 PDB crystals from RCSB + 4 AlphaFold models from EBI (~10 MB).
Idempotent — already-downloaded files are skipped.


In [ ]:
!crypticip download-validation


## Run validation

Iterates the controls, runs fpocket / FreeSASA / APBS where available,
scores each structure, and writes `results/reports/validation/`.

⚠️ Scientific interpretation of the AUC / gate decision requires
fpocket + freesasa + APBS to actually be installed. In fallback mode
the numbers are plumbing diagnostics only.


In [ ]:
!crypticip validate --config config/validation.yaml || echo 'validate exited non-zero (fallback mode or gate failure)'


## Generate the human-readable report


In [ ]:
!crypticip report --validation


## Inspect ADAR2 diagnostics


In [ ]:
import json, pathlib, pandas as pd

vpath = pathlib.Path('results/reports/validation/validation_results.json')
if not vpath.exists():
    print('No validation_results.json yet — run `crypticip validate` first.')
else:
    data = json.loads(vpath.read_text())
    rows = data.get('per_structure') or data.get('records') or []
    df = pd.DataFrame(rows)
    print('records:', len(df))
    cols = [c for c in ['pdb_id','name','label','score','volume_A3',
                        'depth_A','potential_kT_e','sasa_status',
                        'apbs_status'] if c in df.columns]
    if cols:
        print(df[cols].to_string(index=False))


## What to download from Colab

The cell below zips `results/reports/validation/` and downloads it.


In [ ]:
import shutil, pathlib
src = pathlib.Path('results/reports/validation')
if src.exists():
    out = shutil.make_archive('/content/crypticip_validation', 'zip', src)
    print('zipped to:', out, 'MB:', round(pathlib.Path(out).stat().st_size / 1e6, 2))
    try:
        from google.colab import files
        files.download(out)
    except Exception as e:
        print('skipping files.download (not on Colab):', e)
else:
    print('No results/reports/validation/ yet — run the cells above first.')


## Troubleshooting

| Symptom | Fix |
|---|---|
| `fpocket: command not found` | Re-run the *Install external scientific binaries* section with `INSTALL_EXTERNAL=True`, then re-run the bootstrap cell. |
| `validate` exits 1 | Expected in fallback mode — gate threshold isn't met without real pocket detection. |
| `RuntimeError: ... in /content/...` | Probably means cwd drifted; re-run the *Clone + install* cell. |
